In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext memory_profiler

import os
import sys
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import pandas as pd
import scipy as sc
import scanpy as scanpy
from tqdm import tqdm
from sklearn.decomposition import PCA
import umap
from scipy import stats

sys.path.insert(1, '/Users/irwanij/Documents/GitHub/MO2GP')
sys.path.insert(1, '/Users/irwanij/Documents/GitHub/MO2GP/code')

from copy import deepcopy as dcp

import time
import pickle
import anndata as ad
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples

from joblib import Parallel, delayed
from tqdm import tqdm

np.set_printoptions(formatter={'float': lambda x: "{0:0.20f}".format(x)})
font = {'family' : 'arial',
        'weight' : 'bold',
        'size'   : 20}
plt.rc('font', **font)

## Prepare the colormap
custom_colors = [
    "#0000FF", "#FF0000", "#00FF00", "#00002C", "#FF1AB9", "#FFD300", "#005800",
    "#8484FF", "#9E4F46", "#00FFC1", "#008495", "#00007B", "#95D34F", "#F69EDC",
    "#D312FF", "#7B1A6A", "#F61261", "#FFC184", "#232309", "#8DA77B", "#F68409",
    "#847200", "#72F6FF", "#9EC1FF", "#72617B", "#9E0000", "#004FFF", "#004695",
    "#D3FF00", "#B94FD3", "#3E001A", "#EDFFB0", "#FF7B61", "#46FF7B", "#12A761",
    "#D3A7A7", "#D34F84", "#6A00C1", "#2C6146", "#0095F6", "#093E4F", "#A75809",
    "#72613E", "#099500", "#9E6AB9", "#FFFF72", "#A7F6CA", "#95B0B9", "#B0B009",
    "#2C004F", "#00CAFF", "#4F2300", "#00B9A7", "#9E0035", "#4F7BB0", "#1A46C1",
    "#58D300", "#729535", "#E5A735", "#F68D95", "#6A128D", "#E561C1", "#EDD3FF",
    "#9EFF9E"
]
custom_cmap = mcolors.ListedColormap(custom_colors, name="custom_cmap")

In [ ]:
from PIL import Image
import numpy as np
import cv2

def get_image_and_contours(img_path, size=128, threshold=128, invert_background=False):
    try:
        # Open image and convert to grayscale
        img = Image.open(img_path).convert("L")  # "L" for grayscale
    except IOError:
        raise FileNotFoundError(f"Cannot load {img_path}")
    
    # Convert to OpenCV format (numpy array)
    img = np.array(img)

    # Normalize the image to the range [0, 255] if it's not already in uint8 format
    if img.dtype != np.uint8:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # Invert the background if it is bright (background needs to be 0)
    if invert_background:
        img = 255 - img

    # Apply binary thresholding to segment the leaf
    _, binary = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)

    # Get original dimensions
    h, w = binary.shape
    # Determine the scaling factor based on the largest dimension
    scale = size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    # Resize while maintaining aspect ratio
    img_resized = cv2.resize(binary, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    # Create a 128x128 black image for padding
    img_padded = np.zeros((size, size), dtype=np.uint8)
    # Calculate padding
    top = (size - new_h) // 2
    bottom = size - new_h - top
    left = (size - new_w) // 2
    right = size - new_w - left
    # Apply padding to the resized image
    img_padded[top:top + new_h, left:left + new_w] = img_resized

    # Find the largest continuous contour (external shape)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    largest_contour = None
    if contours:
        # Select the largest contour
        largest_contour = max(contours, key=cv2.contourArea)
        largest_contour = largest_contour.reshape(-1, 2)
    
    return img_padded, largest_contour

def my_silhouette_score(dataIn, labels, metric='euclidean'):    
    output_sample = silhouette_samples(dataIn, labels, metric=metric)
    unique_labels = np.unique(labels)
    group_means = np.array([output_sample[labels == label].mean(axis=0) for label in unique_labels])
    return np.mean(group_means)

In [ ]:
from tqdm import tqdm
contour_input = []
labels = []
img_input = []
# Define the path
path = '/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/downloaded_data/MPEG7_dataset/data/'
# Get sorted list of folders (excluding hidden ones)
files = [f for f in os.listdir(path) if not f.startswith('.')]
files = np.sort(files)

for file in tqdm(files):
    label = file.split('-')[0]
    labels.append(label)
    img_path = os.path.join(path, file)
    
    cur_img, cur_cont = get_image_and_contours(img_path, size=128, threshold=128, invert_background=False)
    img_input.append(cur_img)
    contour_input.append(cur_cont)
img_input = np.array(img_input)
print(len(contour_input), len(labels), len(img_input))

with open("/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/contour_mpeg7.pkl", "wb") as f:
    pickle.dump(contour_input, f)
with open("/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/label_mpeg7.pkl", "wb") as f:
    pickle.dump(labels, f)
np.save(file='/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/image_mpeg7.npy', arr=img_input)

In [ ]:
with open("/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/contour_mpeg7.pkl", "rb") as f:
    contour_input = pickle.load(f)
with open("/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/label_mpeg7.pkl", "rb") as f:
    labels = pickle.load(f)
labels = np.array(labels)
img_input = np.load(file='/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/image_mpeg7.npy')

fig, ax = plt.subplots(ncols=10, nrows=7, figsize=(50, 35))
ax = ax.flatten()

fig, ax2 = plt.subplots(ncols=10, nrows=7, figsize=(50, 35))
ax2 = ax2.flatten()

id_ = np.arange(0,len(contour_input),20)
ino = 0
for i in tqdm(id_):
    img = img_input[i]
    c = contour_input[i]

    ax[ino].imshow(img)
    ax[ino].set_title(labels[i])
    ax2[ino].plot(c[:,0], c[:,1])
    
    ino = ino + 1

In [ ]:
from align import ShapeAlign

with open("/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/contour_mpeg7.pkl", "rb") as f:
    contour_input = pickle.load(f)
with open("/Users/irwanij/Documents/GitHub/mogec-code/MOGEC/label_mpeg7.pkl", "rb") as f:
    labels = pickle.load(f)
labels = np.array(labels)
print(len(contour_input), len(labels))

start = time.time()
# Init
model_align = ShapeAlign(contours = contour_input)
# Preprocess contours - centering, interp, smoothing
model_align.preprocess_contours(num_workers=8, n_interp=250, n_smooth=0, scale='perimeter')
# Compute functions of the contour
model_align.get_embedding(num_workers=1) #pcs=None, get_descriptor=True, kernel=1
print('Running time: ', time.time() - start)


shape_embedding = model_align.shape_embedding
contours = model_align.contours

ss = my_silhouette_score(shape_embedding, labels, metric='euclidean')
print(ss, shape_embedding.shape)
